<font color="#CA0032"><h1>Practica B3-T5 - Redes neuronales con entradas heterogeneas</h1></font>

## Version AFINADA - Prediccion de ventas (Rossmann Store Sales)

**Grupo:** Alonso Diaz - Raul Rodriguez - Piettro Enrico

---

Esta es la version **afinada** del modelo. Respecto a la version base
(`notebook_entregable_rossmann.ipynb`) incorpora:

1. **Canales exogenos en la secuencia recurrente**: la LSTM no solo ve el pasado de las ventas, sino tambien
   `Promo`, `SchoolHoliday` y `Open` de cada dia de la ventana.
2. **Variables derivadas (feature engineering)**: lags semanales (`lag_7`, `lag_14`), media y desviacion de
   la ventana, y nº de dias con promocion en la ventana.
3. **Arquitectura mas profunda**: LSTM apilada (2 capas) + bloque denso para las numericas + embeddings.
4. **Busqueda de hiperparametros** (pequeño barrido) con tabla de resultados.
5. **Metrica RMSPE** (la oficial de la competicion) ademas del R2.

**Enunciado del profesor.** Predecir las ventas de las **9 tiendas** `[1, 2, 3, 4, 5, 562, 682, 733, 769]`
maximizando el **R2 en test**. El **test son los datos a partir del 2015-01-01 (incluido)** y el resto es
train/validacion. **No se usa la variable `Customers`** (prohibida por estar trivialmente correlacionada con
las ventas). Esquema *many-to-one*: para predecir el dia `t` la red mira los `W` dias anteriores.

In [ ]:
COLAB = False
RUTA_DATA = 'data'

# Tiendas objetivo del enunciado (5 que cierran domingos + 4 que no cierran nunca)
TIENDAS = [1, 2, 3, 4, 5, 562, 682, 733, 769]

# Estrategia de entrenamiento:
#   False -> entrena SOLO con las 9 tiendas objetivo (rapido, viable en CPU)
#   True  -> modelo global con las 1.115 tiendas (suele mejorar el R2; requiere GPU/Colab)
ENTRENAR_CON_TODAS = False

In [ ]:
import numpy as np, pandas as pd, zipfile, os
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Flatten, concatenate, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

SEED = 7
np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow', tf.__version__)

In [ ]:
if COLAB:
    if not os.path.exists('B3_T5-NN-con-entrdas-hetereogeneas-'):
        !git clone https://github.com/alonsodt/B3_T5-NN-con-entrdas-hetereogeneas-.git
    RUTA_DATA = 'B3_T5-NN-con-entrdas-hetereogeneas-/data'

## 1. Carga, union y variables

In [ ]:
with zipfile.ZipFile(os.path.join(RUTA_DATA, 'train.zip')) as z:
    train = pd.read_csv(z.open('train.csv'), parse_dates=['Date'], low_memory=False)
store = pd.read_csv(os.path.join(RUTA_DATA, 'store.csv'))
test  = pd.read_csv(os.path.join(RUTA_DATA, 'test.csv'), parse_dates=['Date'])

df = train.merge(store, on='Store', how='left').sort_values(['Store', 'Date']).reset_index(drop=True)
df['StateHoliday'] = df['StateHoliday'].astype(str).replace('0', 'n')

df['Month'] = df.Date.dt.month
df['CompetitionDistance'] = df['CompetitionDistance'].fillna(df['CompetitionDistance'].median())
df['CompDistLog'] = np.log1p(df['CompetitionDistance'])
df['Promo2'] = df['Promo2'].fillna(0).astype(int)
df['y'] = np.log1p(df['Sales'])   # objetivo en escala log
print('df:', df.shape, '| continuidad por tienda verificada (sin huecos internos)')

**Tiendas objetivo (enunciado):** `[1, 2, 3, 4, 5, 562, 682, 733, 769]`. Las 5 primeras son tiendas de
barrio que **cierran los domingos**; las 4 ultimas **no cierran nunca** (perfil tipo centro comercial). El
R2 se reporta de forma **global** y restringido a estas tiendas (que es donde evalua el profesor).

In [ ]:
# Tiendas objetivo definidas en la configuracion (celda 1)
print('Tiendas objetivo:', TIENDAS)

# Si no entrenamos el modelo global, restringimos el dataset a las tiendas objetivo (mucho mas rapido)
if not ENTRENAR_CON_TODAS:
    df = df[df.Store.isin(TIENDAS)].reset_index(drop=True)
print('Tiendas en df:', df.Store.nunique(), '| filas:', len(df))

In [ ]:
# Codificacion de categoricas a indices para los embeddings
cat_cols = ['Store', 'StoreType', 'Assortment', 'StateHoliday', 'DayOfWeek', 'Month']
cat_maps, cat_card = {}, {}
for col in cat_cols:
    cats = sorted(df[col].unique())
    cat_maps[col] = {v: i for i, v in enumerate(cats)}
    df[col + '_idx'] = df[col].map(cat_maps[col]).astype(int)
    cat_card[col] = len(cats)
cat_card

## 2. Enventanado afinado (por tienda)

Para cada dia objetivo `t` construimos:
- `seq` (`W` x 4): pasado de `[log_sales, Promo, SchoolHoliday, Open]` -> entra a la LSTM.
- `num`: exogenas del dia `t` + variables derivadas (`lag_7`, `lag_14`, media/desv. de la ventana, nº de
  dias con promo en la ventana).
- categoricas del dia `t` (Store, StoreType, Assortment, StateHoliday, DayOfWeek, Month) -> embeddings.

In [ ]:
W = 28  # 4 semanas de historia (permite lag_7 y lag_14)
CAT_IDX = ['Store_idx', 'StoreType_idx', 'Assortment_idx', 'StateHoliday_idx', 'DayOfWeek_idx', 'Month_idx']
NUM_NAMES = ['Promo', 'SchoolHoliday', 'Promo2', 'CompDistLog', 'lag7', 'lag14', 'mean_w', 'std_w', 'promo_w']

def construir_ventanas(data, W):
    seq, ynum, y_out, fecha, store_id, open_t = [], [], [], [], [], []
    cats = {ccol: [] for ccol in CAT_IDX}
    for _, g in data.groupby('Store'):
        g = g.sort_values('Date')
        yv = g['y'].values; pr = g['Promo'].values; sh = g['SchoolHoliday'].values; op = g['Open'].values
        cd = g['CompDistLog'].values; p2 = g['Promo2'].values
        cat_v = {ccol: g[ccol].values for ccol in CAT_IDX}
        for i in range(W, len(g)):
            wy = yv[i - W:i]
            seq.append(np.stack([wy, pr[i - W:i], sh[i - W:i], op[i - W:i]], axis=1))
            ynum.append([pr[i], sh[i], p2[i], cd[i], yv[i - 7], yv[i - 14],
                         wy.mean(), wy.std(), pr[i - W:i].sum()])
            y_out.append(yv[i]); fecha.append(g['Date'].values[i])
            store_id.append(g['Store'].values[i]); open_t.append(op[i])
            for ccol in CAT_IDX:
                cats[ccol].append(cat_v[ccol][i])
    out = {'seq': np.array(seq, dtype='float32'),
           'num': np.array(ynum, dtype='float32'),
           'y': np.array(y_out, dtype='float32'),
           'fecha': np.array(fecha), 'store': np.array(store_id), 'open': np.array(open_t)}
    for ccol in CAT_IDX:
        out[ccol] = np.array(cats[ccol], dtype='int32')
    return out

ds = construir_ventanas(df, W)
mask = ds['open'] == 1   # entrenamos solo con dias abiertos como objetivo
for k in list(ds.keys()):
    ds[k] = ds[k][mask]
print('muestras:', ds['y'].shape[0], '| seq:', ds['seq'].shape, '| num:', ds['num'].shape)

## 3. Particion temporal y escalado

Segun el enunciado, **test = datos desde 2015-01-01 (incluido)**. Reservamos las ultimas ~6 semanas de 2014
como **validacion** (para *early stopping* y seleccion de modelo) y el resto es **train**. El escalado se
ajusta **solo con train**.

In [ ]:
F_TEST = np.datetime64('2015-01-01'); F_VAL = np.datetime64('2014-11-15')
fechas = ds['fecha']
i_tr = np.where(fechas < F_VAL)[0]
i_va = np.where((fechas >= F_VAL) & (fechas < F_TEST))[0]
i_te = np.where(fechas >= F_TEST)[0]
print('train', len(i_tr), '| val', len(i_va), '| test', len(i_te))

# Escalado: canal 0 de la secuencia (log_sales) y las numericas, con estadisticos de train
esc_seq = StandardScaler().fit(ds['seq'][i_tr][:, :, 0].reshape(-1, 1))
esc_num = StandardScaler().fit(ds['num'][i_tr])

def escala_seq(s):
    s = s.copy()
    sh = s[:, :, 0].shape
    s[:, :, 0] = esc_seq.transform(s[:, :, 0].reshape(-1, 1)).reshape(sh)
    return s.astype('float32')

seq_s = escala_seq(ds['seq'])
num_s = esc_num.transform(ds['num']).astype('float32')

def inputs(idx):
    X = {'seq': seq_s[idx], 'num': num_s[idx]}
    for ccol in CAT_IDX:
        X[ccol] = ds[ccol][idx]
    return X

y_tr, y_va, y_te = ds['y'][i_tr], ds['y'][i_va], ds['y'][i_te]
ventas_te = np.expm1(y_te); store_te = ds['store'][i_te]

## 4. Metricas: R2 y RMSPE

In [ ]:
def rmspe(real, pred):
    m = real > 0
    return np.sqrt(np.mean(((real[m] - pred[m]) / real[m]) ** 2))

def metricas(real, pred, stores):
    sel = np.isin(stores, TIENDAS)
    return {'R2_global': r2_score(real, pred), 'R2_tiendas': r2_score(real[sel], pred[sel]),
            'RMSPE_global': rmspe(real, pred), 'RMSPE_tiendas': rmspe(real[sel], pred[sel])}

## 5. Baseline: media por (tienda, dia de la semana)

In [ ]:
df_tr_open = df[(df.Date < F_TEST) & (df.Open == 1)]
media_td = df_tr_open.groupby(['Store', 'DayOfWeek'])['Sales'].mean()
inv_dow = {v: k for k, v in cat_maps['DayOfWeek'].items()}
dow_te = np.array([inv_dow[i] for i in ds['DayOfWeek_idx'][i_te]])
pred_base = np.array([media_td.get((s, d), df_tr_open['Sales'].mean()) for s, d in zip(store_te, dow_te)])
print('Baseline:', {k: round(v, 4) for k, v in metricas(ventas_te, pred_base, store_te).items()})

## 6. Modelo afinado con entradas heterogeneas

LSTM apilada para la secuencia + embeddings para las categoricas + bloque denso para las numericas.

In [ ]:
def construir_modelo(u1=64, u2=32, dropout=0.3, dim_emb=10, dense=128):
    seq_in = Input(shape=(W, 4), name='seq')
    h = LSTM(u1, return_sequences=True)(seq_in)
    h = LSTM(u2)(h)
    ramas = [h]
    cat_inputs = []
    for ccol in CAT_IDX:
        card = cat_card[ccol.replace('_idx', '')]
        d = int(min(dim_emb, max(2, card // 2)))
        inp = Input(shape=(1,), name=ccol)
        cat_inputs.append(inp)
        ramas.append(Flatten()(Embedding(card, d)(inp)))
    num_in = Input(shape=(len(NUM_NAMES),), name='num')
    ramas.append(Dense(32, activation='relu')(num_in))
    x = concatenate(ramas)
    x = Dense(dense, activation='relu')(x)
    x = Dropout(dropout)(x)
    x = Dense(dense // 2, activation='relu')(x)
    out = Dense(1)(x)
    model = Model(inputs=[seq_in] + cat_inputs + [num_in], outputs=out)
    model.compile(loss='mse', optimizer='adam', metrics=['mae'])
    return model

In [ ]:
def entrenar(model, epochs=40, batch_size=512, nombre='modelo'):
    cbs = [ModelCheckpoint(nombre + '.keras', monitor='val_loss', save_best_only=True),
           EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)]
    return model.fit(inputs(i_tr), y_tr, validation_data=(inputs(i_va), y_va),
                     epochs=epochs, batch_size=batch_size, verbose=2, callbacks=cbs)

def evaluar(model):
    pred = np.clip(np.expm1(model.predict(inputs(i_te), verbose=0).flatten()), 0, None)
    return metricas(ventas_te, pred, store_te)

## 7. Busqueda de hiperparametros

Pequeño barrido sobre tamaño de las LSTM, dropout y dimension de embeddings. Nos quedamos con la mejor
configuracion segun el **R2 en las 10 tiendas** del test interno. (Subir `epochs` al ejecutar de verdad.)

In [ ]:
configs = [
    dict(u1=64,  u2=32, dropout=0.3, dim_emb=10, dense=128),
    dict(u1=96,  u2=48, dropout=0.3, dim_emb=16, dense=256),
    dict(u1=128, u2=64, dropout=0.4, dim_emb=16, dense=256),
]
resultados = []
mejor, mejor_r2, mejor_cfg = None, -np.inf, None
for k, cfg in enumerate(configs):
    tf.random.set_seed(SEED)
    m = construir_modelo(**cfg)
    entrenar(m, epochs=40, nombre=f'afinado_{k}')
    met = evaluar(m)
    resultados.append({**cfg, **{kk: round(vv, 4) for kk, vv in met.items()}})
    print(cfg, '->', {kk: round(vv, 4) for kk, vv in met.items()})
    if met['R2_tiendas'] > mejor_r2:
        mejor, mejor_r2, mejor_cfg = m, met['R2_tiendas'], cfg
tabla = pd.DataFrame(resultados)
print('\nMejor configuracion:', mejor_cfg)
tabla

## 8. Comparativa final (test = 2015)

In [ ]:
comp = pd.DataFrame([
    {'Modelo': 'Baseline media (tienda,dia)', **{k: round(v, 4) for k, v in metricas(ventas_te, pred_base, store_te).items()}},
    {'Modelo': 'Afinado (mejor config)',      **{k: round(v, 4) for k, v in evaluar(mejor).items()}},
])
comp

## 9. Visualizacion de embeddings (dia de la semana)

In [ ]:
emb_layers = [l for l in mejor.layers if l.name.startswith('embedding')]
emb_por_cat = dict(zip(CAT_IDX, emb_layers))  # creados en el orden de CAT_IDX
Wd = emb_por_cat['DayOfWeek_idx'].get_weights()[0]
print('shape embedding DayOfWeek:', Wd.shape)
if Wd.shape[1] >= 2:
    nombres = ['lun', 'mar', 'mie', 'jue', 'vie', 'sab', 'dom']
    plt.figure(figsize=(5, 4))
    plt.scatter(Wd[:, 0], Wd[:, 1])
    for i, n in enumerate(nombres[:Wd.shape[0]]):
        plt.text(Wd[i, 0], Wd[i, 1], n)
    plt.title('Embedding del dia de la semana'); plt.grid(); plt.show()

## 10. Reflexion final

*(Rellenar con los numeros obtenidos tras ejecutar.)*

- **Que aporta afinar.** Los canales exogenos en la secuencia (que la LSTM vea las promos pasadas) y los
  lags semanales (`lag_7`, `lag_14`) capturan la fuerte estacionalidad semanal de Rossmann, que es donde
  mas se gana en R2. El embedding de `Store` permite un unico modelo para todas las tiendas objetivo.
- **Las dos familias de tiendas.** Las 5 que cierran domingos son muy periodicas (R2 alto esperable); las 4
  que no cierran nunca son mas irregulares y suelen dar un R2 menor. Conviene mirar el R2 por tienda.
- **R2 vs RMSPE.** El R2 mide varianza explicada; el RMSPE (metrica oficial Kaggle) penaliza el error
  relativo. Reportar ambos da una vision mas honesta del rendimiento por tienda.
- **Decisiones de diseño.** Objetivo en escala `log1p(Sales)`; entrenamos solo con dias abiertos
  (`Open==1`); no usamos `Customers` (prohibida); particion temporal estricta (test = 2015).
- **Lineas de mejora.** Modelo global con las 1.115 tiendas (`ENTRENAR_CON_TODAS=True`) para transferir
  conocimiento entre tiendas; un modelo por familia (cierran vs no cierran); ensemble de semillas; tuning
  mas amplio (Optuna/Keras-Tuner); calendario de festivos del estado aleman correspondiente.